In [1]:
import polars as pl
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error

from catboost import CatBoostRegressor

import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/ya_realty_with_txt_embeds.parquet')
df.shape

(62410, 23)

In [3]:
df.head()

,offer_id,price,price_numeric,old_price,area,rooms,floor,price_per_m2,metro,metro_time,...,photo_count,badges,publish_date,url,title,description,image_urls,self_floor,max_floor,description_embedding
0,7035113340557126091,7 500 000 ₽,7500000.0,NaN,17.7,студия,9 этаж из 16,None,Калитники,9.0,...,1.0,None,None,https://realty.yandex.ru/offer/703511334055712...,апартаменты-студия,Номер лота: 99696. Панорамный вид из больших о...,https://avatars.mds.yandex.net/get-realty-offe...,16.0,16.0,"[0.018224586, 0.05090819, -0.050330937, -0.052..."
1,7035113340416809607,7 500 000 ₽,7500000.0,NaN,17.0,студия,2 этаж из 2,None,Соколиная гора,8.0,...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/703511334041680...,апартаменты-студия,Номер лота: 87440. Продается студия с дизайнер...,https://avatars.mds.yandex.net/get-realty-offe...,2.0,2.0,"[0.03725325, 0.049842387, -0.0155871445, -0.04..."
2,7053956964805047621,12 200 000 ₽,12200000.0,NaN,17.9,студия,2 этаж из 48,None,Тушинская,10.0,...,1.0,None,3 квартал 2027,https://realty.yandex.ru/offer/705395696480504...,квартира-студия,"Арт. 119802099 Студия 17,9 м в CITYZEN Урбан-б...",https://avatars.mds.yandex.net/get-realty-offe...,48.0,48.0,"[0.016696937, 0.059451837, -0.033862226, -0.05..."
3,7053956914445503237,7 300 000 ₽,7300000.0,NaN,15.7,студия,5 этаж из 5,None,Бутырская,17.0,...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/705395691444550...,квартира-студия,Арт. 134010491 СПЕЦИАЛЬНО для наших клиентов с...,https://avatars.mds.yandex.net/get-realty-offe...,5.0,5.0,"[0.01692044, 0.038449533, -0.046054967, -0.066..."
4,3699730400767130013,10 802 031 ₽,10802031.0,NaN,14.1,студия,5 этаж из 16,None,Коммунарка,14.0,...,1.0,None,2 квартал 2026,https://realty.yandex.ru/offer/369973040076713...,квартира-студия,Строим кварталы для жизни с заботой о будущем....,https://avatars.mds.yandex.net/get-realty-offe...,16.0,16.0,"[-0.0043280213, 0.04248244, -0.020798901, -0.0..."


In [ ]:
X = df[['area', 'metro_time', 'photo_count', 
'self_floor', 'max_floor', 'description_embedding']]

Y = df['price_numeric']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, shuffle=True)

In [ ]:
model = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=5,
    l2_leaf_reg=1,
    # cat_features=['metro', 'title', 'author'],
    # text_features=['description', 'address', 'metro'],
    embedding_features=['description_embedding'],
    verbose=200
)

model.fit(X_train, Y_train)

In [ ]:
def eval_with_metrics(model, X, Y):
    
    preds = model.predict(X)

    print(f"R^2: {r2_score(Y, preds)} \n"
          f"MAE: {mean_absolute_error(Y, preds)} \n"
          f"MAPE: {mean_absolute_percentage_error(Y, preds)} \n"
          f"RMSE: {root_mean_squared_error(Y, preds)} \n")
    
eval_with_metrics(model, X_val, Y_val)

In [4]:
X = df[['area', 'metro_time', 'photo_count', 
'self_floor', 'max_floor', 'metro', 'title', 
'author', 'description', 'address', 'description_embedding']]

X['metro'] = X['metro'].apply(lambda x: str(x))
X['title'] = X['title'].apply(lambda x: str(x))
X['author'] = X['author'].apply(lambda x: str(x))
X['description'] = X['description'].apply(lambda x: str(x))
X['address'] = X['address'].apply(lambda x: str(x))


Y = df['price_numeric']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.1, shuffle=True)

In [ ]:
model_full = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=5,
    l2_leaf_reg=1,
    cat_features=['metro', 'title', 'author'],
    text_features=['description', 'address', 'metro'],
    embedding_features=['description_embedding'],
    verbose=200
)

model_full.fit(X_train, Y_train)

In [17]:
best_params = {'learning_rate': 0.06716416451757115, 
 'depth': 6, 
 'l2_leaf_reg': 3.432744002009776e-06, 
 'random_strength': 6.668427569822431e-05, 
 'border_count': 40, 
 'grow_policy': 'Lossguide', 
 'bootstrap_type': 'Bernoulli', 
 'subsample': 0.8574901373130458, 
 'min_data_in_leaf': 4, 
 'max_leaves': 17,
 'max_ctr_complexity': 4,
 'one_hot_max_size': 22,
 
'iterations': 5000,
"loss_function": "RMSE",
"eval_metric": "RMSE",
"od_type": "Iter",
"od_wait": 50,
"verbose": 100,
"random_state": 42,
}

embedding_feature_options = {
'calcers': ['KNN:k=4,distance_type=Cosine',
                                         'KNN:k=100,distance_type=Cosine',
                                         'LDA:output_dim:10']
}

In [18]:
model = CatBoostRegressor(
**best_params,
cat_features=['title', 'author'],
text_features=['description', 'address', 'metro'],
embedding_features=['description_embedding'],

)

model.fit(X_train, Y_train, eval_set=[(X_val, Y_val)], early_stopping_rounds=100)

0:	learn: 38943597.4220134	test: 43723097.3275805	best: 43723097.3275805 (0)	total: 440ms	remaining: 36m 41s
100:	learn: 14441435.6147139	test: 17867823.1581726	best: 17867823.1581726 (100)	total: 21.9s	remaining: 17m 43s
200:	learn: 12184816.7271474	test: 16254850.5699345	best: 16254850.5699345 (200)	total: 40.8s	remaining: 16m 14s
300:	learn: 10995110.7034451	test: 15504670.3131110	best: 15504670.3131110 (300)	total: 59.5s	remaining: 15m 29s
400:	learn: 10110368.8084836	test: 15096025.9608758	best: 15096025.9608758 (400)	total: 1m 18s	remaining: 14m 56s
500:	learn: 9492058.8346445	test: 14854318.1153907	best: 14854318.1153907 (500)	total: 1m 37s	remaining: 14m 34s
600:	learn: 8963755.5144320	test: 14609696.4826471	best: 14609696.4826471 (600)	total: 1m 57s	remaining: 14m 19s
700:	learn: 8535057.3025564	test: 14493112.7266897	best: 14493112.7266897 (700)	total: 2m 17s	remaining: 14m 3s
800:	learn: 8160588.1567884	test: 14363263.1255573	best: 14362880.9631060 (799)	total: 2m 36s	remain

CatBoostRegressor(bootstrap_type='Bernoulli', border_count=40, cat_features=['title', 'author'], depth=6, embedding_features=['description_embedding'], eval_metric='RMSE', grow_policy='Lossguide', iterations=5000, l2_leaf_reg=3.432744002009776e-06, learning_rate=0.06716416451757115, loss_function='RMSE', max_ctr_complexity=4, max_leaves=17, min_data_in_leaf=4, od_type='Iter', od_wait=50, one_hot_max_size=22, random_state=42, random_strength=6.668427569822431e-05, subsample=0.8574901373130458, text_features=['description', 'address', 'metro'], verbose=100)

In [19]:
model.save_model("catb_text_60K.cbm")

In [20]:
def eval_with_metrics(model, X, Y):
    
    preds = model.predict(X)

    print(f"R^2: {r2_score(Y, preds)} \n"
          f"MAE: {mean_absolute_error(Y, preds)} \n"
          f"MAPE: {mean_absolute_percentage_error(Y, preds)} \n"
          f"RMSE: {root_mean_squared_error(Y, preds)} \n")
    
eval_with_metrics(model, X_val, Y_val)

R^2: 0.911144318847452 
MAE: 4424822.148812117 
MAPE: 0.1581866620150293 
RMSE: 13567711.3975676 

